# ViViT video-to-text experiment

Recreated from `Vivit2.0_fff.ipynb`. Shared model definitions are imported from `src/sign_language_lab/`; training and evaluation remain visible below. Outputs were cleared during organization.

**Status:** historical experiment with unresolved model and evaluation issues. Formatting and module extraction do not validate its results. Read [the experiment review](../docs/experiment_review.md) and [setup guide](../docs/reproduction.md) before executing cells.

Cell metadata records original cell numbers. Dataset and checkpoint paths now use environment-based configuration.

## Repository and path setup

In [ ]:
import sys
from pathlib import Path

repository_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "sign_language_lab").is_dir()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
source_directory = str(repository_root / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from sign_language_lab.config import ExperimentPaths

paths = ExperimentPaths.from_environment("vivit_translation")
paths.output_dir.mkdir(parents=True, exist_ok=True)


## Hardware and imports

In [ ]:
import torch as torch


In [ ]:
torch.cuda.is_available()


In [ ]:
print(torch.cuda.get_device_name(0))


In [ ]:
import av
import numpy as np
import torch


## Inspect the pretrained video backbone

In [ ]:
from transformers import VivitModel

model = VivitModel.from_pretrained(
    "google/vivit-b-16x2-kinetics400",
    attn_implementation="sdpa",
    torch_dtype=torch.float16,
)


In [ ]:
# Convert the model to float32 before summarizing
from torchinfo import summary


## Locate input videos

In [ ]:
import os
import glob

train_dir = str(paths.train_dir)
# List all video files in the folder (adjust extension as necessary)
video_files = glob.glob(
    os.path.join(train_dir, "*.mp4")
)  # You can also try '*.avi', '*.mov', etc.

if video_files:
    print("Found the following video files:")
    for video_file in video_files:
        # print(video_file)
        count = 1
else:
    print("No video files found in the directory.")


In [ ]:
import cv2

video_path = video_files[0]  # Use the first video file from the list

# Open the video file
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Could not open video {video_path}")
else:
    print(f"Successfully opened video: {video_path}")
    # Read the first frame
    ret, frame = cap.read()
    if ret:
        print("First frame read successfully.")
    else:
        print("Error: Could not read the first frame.")
    cap.release()  # Release the video capture object


## Video preprocessing

In [ ]:
import av
import cv2
import torch
import numpy as np
from typing import List


### `sample_frame_indices`

In [ ]:
def sample_frame_indices(
    clip_len: int, frame_sample_rate: int, seg_len: int
) -> np.ndarray:
    """
    Generate frame indices making sure we don't get the low >= high error.
    """
    converted_len = int(clip_len * frame_sample_rate)

    # Make sure we have enough frames
    if seg_len <= converted_len:
        # If we don't have enough frames, just use all frames with repetition
        indices = np.linspace(0, seg_len - 1, num=clip_len)
    else:
        # If we have enough frames, sample randomly
        end_idx = seg_len - 1  # Use the last frame as the maximum
        start_idx = max(0, end_idx - converted_len)  # Ensure start_idx is not negative
        indices = np.linspace(start_idx, end_idx, num=clip_len)

    return indices.astype(np.int64)


### `read_video_pyav`

In [ ]:
def read_video_pyav(
    container, indices: np.ndarray, target_size=(224, 224)
) -> np.ndarray:
    """
    Read video frames using PyAV and resize them with OpenCV.
    """
    frames = []
    container.seek(0)

    # Convert indices to list for checking membership
    indices_list = indices.tolist()

    for i, frame in enumerate(container.decode(video=0)):
        if i in indices_list:
            # Convert to numpy array
            img = frame.to_ndarray(format="rgb24")

            # Resize using cv2
            img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)

            # Normalize to [0, 1]
            img_normalized = img_resized.astype(np.float32) / 255.0

            frames.append(img_normalized)

            # Break if we have collected all frames we need
            if len(frames) == len(indices):
                break

    # If we couldn't get enough frames, repeat the last frame
    while len(frames) < len(indices):
        frames.append(
            frames[-1]
            if frames
            else np.zeros((target_size[1], target_size[0], 3), dtype=np.float32)
        )

    return np.stack(frames)


### `process_video_batch`

In [ ]:
def process_video_batch(
    video_paths: List[str], clip_len: int = 32, frame_sample_rate: int = 1
) -> torch.Tensor:
    """
    Process multiple videos into a batch.
    """
    batch_frames = []

    for video_path in video_paths:
        try:
            # Open video container
            container = av.open(video_path)

            # Get total number of frames
            total_frames = container.streams.video[0].frames

            # Get frame indices
            indices = sample_frame_indices(
                clip_len=clip_len,
                frame_sample_rate=frame_sample_rate,
                seg_len=total_frames,
            )

            # Read and resize frames
            frames = read_video_pyav(container, indices, target_size=(224, 224))

            # Close container
            container.close()

            # Add to batch
            batch_frames.append(frames)

        except Exception as e:
            print(f"Error processing video {video_path}: {str(e)}")
            continue

    # Stack all videos into a batch
    if not batch_frames:
        raise ValueError("No videos were successfully processed")

    # Convert to torch tensor and permute dimensions to [B, T, C, H, W]
    batch_tensor = torch.FloatTensor(np.stack(batch_frames))
    batch_tensor = batch_tensor.permute(0, 1, 4, 2, 3)

    # Verify shape
    expected_shape = (len(batch_frames), clip_len, 3, 224, 224)
    assert (
        batch_tensor.shape == expected_shape
    ), f"Incorrect shape: got {batch_tensor.shape}, expected {expected_shape}"

    return batch_tensor


# Example usage:


### `process_batch`

In [ ]:
def process_batch(video_files, batch_size=5):
    """
    Process a batch of videos and return the tensor.
    """
    video_batch = video_files[:batch_size]
    processed_batch = process_video_batch(video_batch, clip_len=32, frame_sample_rate=1)
    print(f"Processed batch shape: {processed_batch.shape}")
    return processed_batch


In [ ]:
# Process your batch of 5 videos
video_files_batch = video_files[:5]
video_batch = process_video_batch(video_files_batch)
print(f"Batch shape: {video_batch.shape}")  # Should be [5, 32, 3, 224, 224]


## Attention components

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import VivitForVideoClassification
from sign_language_lab.attention import (
    HandUpperBodyAttention,
)  # [batch_size, num_frames, feature_dim]

from sign_language_lab.attention import SpatialAttention
from sign_language_lab.vivit_translation import TemporalSkipConnection
from sign_language_lab.attention import PositionalEncoding
from sign_language_lab.attention import EnhancedPositionalEncoding


## Encoder implementation

In [ ]:
from sign_language_lab.vivit_translation import VivitEncoderWithTemporalModel


In [ ]:
model = VivitEncoderWithTemporalModel(
    pretrained_model_name="google/vivit-b-16x2-kinetics400",
    feature_dim=768,
    hidden_dim=512,
    num_layers=2,
    temporal_model="bilstm",
    freeze_encoder=True,
    use_sliding_windows=True,
    window_size=32,
    window_stride=16,
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Create a sample input tensor and move it to the same device
batch_size = 2
input_tensor = torch.randn(batch_size, 32, 3, 224, 224, dtype=torch.float32).to(
    device
)  # Added .to(device) here

# Summarize the model
summary(
    model=model,
    input_data=input_tensor,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    row_settings=["var_names"],
)


In [ ]:
if __name__ == "__main__":
    # Create the encoder with a BiLSTM temporal model
    temporal_encoder = VivitEncoderWithTemporalModel(
        pretrained_model_name="google/vivit-b-16x2-kinetics400",
        temporal_model="transformer",  # Change to "gru" or "transformer" as needed
        freeze_encoder=True,
    )

    # Create a sample input (batch of videos)
    batch_size = 2
    sample_input = torch.randn(batch_size, 32, 3, 224, 224).to(
        next(temporal_encoder.parameters()).device
    )

    # Get encoder outputs
    with torch.no_grad():
        temporal_outputs = temporal_encoder(sample_input)

    print(f"Temporal output shape: {temporal_outputs.shape}")


## Feature extraction and inspection

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from typing import List, Optional, Tuple
import time

# TESTING THE ENCODER PART ONLY


### `test_vivit_on_videos`

In [ ]:
def test_vivit_on_videos(
    model: nn.Module,
    video_paths: List[str],
    batch_size: int = 2,
    device: str = "cuda",
    save_features: bool = False,
    output_dir: Optional[str] = None,
) -> Tuple[torch.Tensor, dict]:
    """
    Test the ViViT encoder with temporal modeling on real video data.

    Args:
        model (nn.Module): The ViViT encoder model with temporal modeling
        video_paths (List[str]): List of paths to video files
        batch_size (int): Number of videos to process in each batch
        device (str): Device to run the model on
        save_features (bool): Whether to save extracted features
        output_dir (Optional[str]): Directory to save features if save_features is True

    Returns:
        Tuple[torch.Tensor, dict]: Extracted features and performance metrics
    """
    print("\n=== Starting ViViT Video Analysis ===")

    # Move model to device and set to eval mode
    model = model.to(device)
    model.eval()

    # Initialize metrics
    metrics = {
        "processing_times": [],
        "memory_usage": [],
        "batch_sizes": [],
        "successful_videos": 0,
        "failed_videos": 0,
    }

    all_features = []

    # Process videos in batches
    for i in range(0, len(video_paths), batch_size):
        batch_paths = video_paths[i : i + batch_size]
        print(
            f"\nProcessing batch {i//batch_size + 1}/{(len(video_paths) + batch_size - 1)//batch_size}"
        )

        try:
            # Time the processing
            start_time = time.time()

            # Process video batch using your existing function
            video_batch = process_video_batch(
                batch_paths, clip_len=32, frame_sample_rate=1
            )
            video_batch = video_batch.to(device)

            print("\n=== Batch Input Verification ===")
            print(f"Batch shape: {video_batch.shape}")
            print(f"Value range: [{video_batch.min():.3f}, {video_batch.max():.3f}]")
            print(f"Successfully processed {len(batch_paths)} videos")

            # Extract features
            with torch.no_grad():
                batch_features = model(video_batch)

            # Analyze features
            print("\n=== Feature Analysis ===")
            print(f"Feature shape: {batch_features.shape}")
            print(
                f"Feature range: [{batch_features.min():.3f}, {batch_features.max():.3f}]"
            )
            print(f"Feature mean: {batch_features.mean():.3f}")
            print(f"Feature std: {batch_features.std():.3f}")

            # Check for numerical stability
            if torch.isnan(batch_features).any():
                print("WARNING: NaN values detected in features!")
            if torch.isinf(batch_features).any():
                print("WARNING: Infinite values detected in features!")

            # Memory tracking
            if torch.cuda.is_available():
                memory_usage = torch.cuda.memory_allocated() / 1024**2
                print(f"\nCUDA Memory Usage: {memory_usage:.2f} MB")
                metrics["memory_usage"].append(memory_usage)

            # Save features if requested
            if save_features and output_dir:
                output_dir = Path(output_dir)
                output_dir.mkdir(parents=True, exist_ok=True)
                for j, path in enumerate(batch_paths):
                    feature_path = output_dir / f"features_batch{i}_video{j}.pt"
                    torch.save(batch_features[j], feature_path)

            # Update metrics
            process_time = time.time() - start_time
            metrics["processing_times"].append(process_time)
            metrics["batch_sizes"].append(len(batch_paths))
            metrics["successful_videos"] += len(batch_paths)

            all_features.append(batch_features.cpu())

            print(f"Batch processing time: {process_time:.2f} seconds")

        except Exception as e:
            print(f"Error processing batch: {str(e)}")
            metrics["failed_videos"] += len(batch_paths)
            continue

    # Combine all features
    if all_features:
        all_features = torch.cat(all_features, dim=0)
    else:
        raise ValueError("No features were successfully extracted")

    # Calculate final metrics
    metrics["total_time"] = sum(metrics["processing_times"])
    metrics["average_time_per_batch"] = np.mean(metrics["processing_times"])
    metrics["average_time_per_video"] = (
        metrics["total_time"] / metrics["successful_videos"]
    )
    if metrics["memory_usage"]:
        metrics["peak_memory_usage"] = max(metrics["memory_usage"])

    print("\n=== Final Processing Statistics ===")
    print(f"Total videos processed: {metrics['successful_videos']}")
    print(f"Failed videos: {metrics['failed_videos']}")
    print(f"Total processing time: {metrics['total_time']:.2f} seconds")
    print(f"Average time per video: {metrics['average_time_per_video']:.2f} seconds")
    if "peak_memory_usage" in metrics:
        print(f"Peak memory usage: {metrics['peak_memory_usage']:.2f} MB")

    return all_features, metrics


### `analyze_temporal_patterns`

In [ ]:
def analyze_temporal_patterns(features: torch.Tensor) -> None:
    """
    Analyze temporal patterns in the extracted features.

    Args:
        features (torch.Tensor): Extracted features of shape [N, T, D]
    """
    print("\n=== Temporal Pattern Analysis ===")

    # Compute frame-to-frame differences
    frame_diffs = torch.norm(features[:, 1:] - features[:, :-1], dim=-1)

    print(f"Average frame-to-frame difference: {frame_diffs.mean():.3f}")
    print(f"Max frame-to-frame difference: {frame_diffs.max():.3f}")
    print(f"Min frame-to-frame difference: {frame_diffs.min():.3f}")

    # Analyze temporal consistency
    temporal_std = features.std(dim=1).mean()
    print(f"Average temporal standard deviation: {temporal_std:.3f}")


# Example usage


In [ ]:
if __name__ == "__main__":
    # Create model instance
    model = VivitEncoderWithTemporalModel(
        pretrained_model_name="google/vivit-b-16x2-kinetics400",
        feature_dim=768,
        hidden_dim=512,
        temporal_model="bilstm",
        freeze_encoder=True,
    )

    # List of video paths
    video_paths = video_files[:5]

    # Extract and analyze features
    features, metrics = test_vivit_on_videos(
        model=model,
        video_paths=video_paths,
        batch_size=2,
        save_features=True,
        output_dir="output/features",
    )

    # Analyze temporal patterns
    analyze_temporal_patterns(features)


## Decoder and model wrapper

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from torch.nn import TransformerDecoder, TransformerDecoderLayer

from sign_language_lab.vivit_translation import SignLanguageDecoder

# Complete model combining encoder and decoder
from sign_language_lab.vivit_translation import SignLanguageTranslationModel

# Loss function
from sign_language_lab.losses import SignLanguageTranslationLoss


## Auxiliary losses and temporal attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from sign_language_lab.losses import LabelSmoothingLoss
from sign_language_lab.losses import ConsistencyLoss
from sign_language_lab.attention import TemporalAttention


In [ ]:
from torch.optim.lr_scheduler import OneCycleLR
from transformers import get_cosine_schedule_with_warmup


## Training annotations

In [ ]:
excel_file = str(paths.annotations_file)
test_dir = str(paths.test_dir)
train_dir = str(paths.train_dir)


In [ ]:
# RETRIVING VIDEO_FILES, AND SENTENCES
import pandas as pd
import glob
import os

video_files1 = glob.glob(
    os.path.join(train_dir, "*.mp4")
)  # Adjust extension as necessary

if video_files1:
    print(f"Found {len(video_files1)} video files.")
else:
    print("No video files found in the directory.")
    video_files1 = []  # Exit early if no video files found

# Step 2: Extract codes from video filenames
video_file_codes1 = [
    os.path.splitext(os.path.basename(video))[0].split("-")[-1].strip()
    for video in video_files1
]

# Step 3: Load Excel file and extract codes from the 'train' sheet
try:
    df_combined = pd.read_excel(
        excel_file, sheet_name="train"
    )  # Adjust sheet name as necessary
    df_combined["code"] = df_combined["name"].apply(lambda x: x.split("-")[-1].strip())

    # Step 4: Match video file codes with codes from the Excel file
    matching_data = df_combined[df_combined["code"].isin(video_file_codes1)]

    if not matching_data.empty:
        matched_translations1 = []
        matched_video_files1 = []

        # Iterate over video file codes to match properly
        for video_code in video_file_codes1:
            matching_row = matching_data[matching_data["code"] == video_code]
            if not matching_row.empty:
                matched_translations1.append(matching_row["translation"].iloc[0])
                matched_video_files1.append(video_code)

    else:
        # Handle cases with no matches
        unmatched_video_codes = [
            code
            for code in video_file_codes2
            if code not in df_combined["code"].tolist()
        ]

except FileNotFoundError:
    print(f"Error: Excel file '{excel_file}' not found.")
except KeyError as e:
    print(f"Error: Missing expected column in Excel file. Details: {e}")
# print((matched_translations1[:100]))
# print(len(video_files1[:5]))


In [ ]:
collected_sentences = matched_translations1[:100]


## Development annotations

In [ ]:
val_dir = str(paths.dev_dir)
val_excel = str(paths.dataset_root / "annotations" / "dev.csv")


In [ ]:
# RETRIVING VIDEO_FILES, AND SENTENCES for validation
import pandas as pd

video_files2 = glob.glob(
    os.path.join(val_dir, "*.mp4")
)  # Adjust extension as necessary

if video_files2:
    print(f"Found {len(video_files2)} video files.")
else:
    print("No video files found in the directory.")
    video_files2 = []  # Exit early if no video files found

# Step 2: Extract codes from video filenames
video_file_codes2 = [
    os.path.splitext(os.path.basename(video))[0].split("-")[-1].strip()
    for video in video_files2
]

# Step 3: Load Excel file and extract codes from the 'train' sheet
try:
    df_combined = pd.read_excel(
        excel_file, sheet_name="dev"
    )  # Adjust sheet name as necessary
    df_combined["code"] = df_combined["name"].apply(lambda x: x.split("-")[-1].strip())

    # Step 4: Match video file codes with codes from the Excel file
    matching_data = df_combined[df_combined["code"].isin(video_file_codes2)]

    if not matching_data.empty:
        matched_translations2 = []
        matched_video_files2 = []

        # Iterate over video file codes to match properly
        for video_code in video_file_codes2:
            matching_row = matching_data[matching_data["code"] == video_code]
            if not matching_row.empty:
                matched_translations2.append(matching_row["translation"].iloc[0])
                matched_video_files2.append(video_code)

    else:
        # Handle cases with no matches
        unmatched_video_codes = [
            code
            for code in video_file_codes2
            if code not in df_combined["code"].tolist()
        ]

except FileNotFoundError:
    print(f"Error: Excel file '{excel_file}' not found.")
except KeyError as e:
    print(f"Error: Missing expected column in Excel file. Details: {e}")
print((matched_translations2[:5]))
print((video_files2[:5]))


In [ ]:
print(len(video_files2))
print(len(matched_translations2))


## Historical training and validation loop

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from tqdm import tqdm
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.model_selection import train_test_split


### `SignLanguageDataset`

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, video_paths, sentences):
        assert len(video_paths) == len(
            sentences
        ), "Number of videos and sentences must match"
        self.video_paths = video_paths
        self.sentences = sentences

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        return {
            "video_path": [
                self.video_paths[idx]
            ],  # Wrap in list for process_video_batch
            "text": self.sentences[idx],
        }


### `collate_fn`

In [ ]:
def collate_fn(batch):
    # Combine all video paths and sentences from the batch
    video_paths = []
    sentences = []

    for item in batch:
        video_paths.extend(item["video_path"])
        sentences.append(item["text"])

    return {"video_path": video_paths, "text": sentences}


### `AverageMeter`

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


### `train_epoch`

In [ ]:
def train_epoch(
    model, dataloader, criterion, optimizer, scheduler, tokenizer, device, epoch
):
    model.train()
    losses = AverageMeter()
    gradient_accumulation_steps = 4
    optimizer.zero_grad()
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch}")
    for batch_idx, batch in pbar:
        try:
            # Process video batch using your function
            batch_videos = process_video_batch(
                batch["video_path"], clip_len=32, frame_sample_rate=1
            )
            batch_videos = batch_videos.to(device)
            texts = batch["text"]

            # Tokenize target text
            tokenized = tokenizer(
                texts, padding=True, truncation=True, return_tensors="pt"
            ).input_ids.to(device)

            # Forward pass
            logits = model(batch_videos, tokenized)

            # Calculate loss (ignore padding tokens)
            loss = criterion(logits[:, :-1].contiguous(), tokenized[:, 1:].contiguous())
            loss = loss / gradient_accumulation_steps
            loss.backward()

            if (batch_idx + 1) % gradient_accumulation_steps == 0 or (
                batch_idx + 1
            ) == len(dataloader):

                optimizer.step()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.zero_grad()

            # Update metrics
            scheduler.step()
            losses.update(
                loss.item() * gradient_accumulation_steps
            )  # Scale back for logging
            pbar.set_postfix({"loss": losses.avg})

        except Exception as e:
            print(f"Error in batch {batch_idx}: {str(e)}")
            continue

    return losses.avg


### `validate`

In [ ]:
@torch.no_grad()
@torch.no_grad()
def validate(model, dataloader, tokenizer, device, beam_size=5, temperature=0.7):
    model.eval()
    all_predictions = []
    all_targets = []

    for batch in tqdm(dataloader, desc="Validating"):
        try:
            videos = process_video_batch(
                batch["video_path"], clip_len=32, frame_sample_rate=1
            )
            videos = videos.to(device)
            texts = batch["text"]

            # Generate translations
            encoder_output = model.encoder(videos)
            predicted_tokens = model.decoder.generate_with_beam_search(
                encoder_output,
                max_length=128,
                beam_size=beam_size,
                temperature=temperature,
                length_penalty=0.7,
            )

            # Convert predictions to text
            predictions = tokenizer.batch_decode(
                predicted_tokens, skip_special_tokens=True
            )
            all_predictions.extend(predictions)
            all_targets.extend(texts)

        except Exception as e:
            print(f"Error during validation: {str(e)}")
            traceback.print_exc()  # Print the full traceback for debugging
            # Add empty predictions for failed batches
            all_predictions.extend([""] * len(texts))
            all_targets.extend(texts)
            continue

    return all_predictions, all_targets


### `main`

In [ ]:
def main():
    # Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Your video paths and sentences
    video_paths = video_files1[:10]
    sentences = matched_translations1[:10]
    val_video_paths = video_files1[:2]
    val_sentences = matched_translations1[:2]

    # val_video_paths=video_files2[:2]
    # val_sentences=matched_translations2[:2]

    # Create dataset and dataloader
    dataset = SignLanguageDataset(video_paths, sentences)
    val_dataset = SignLanguageDataset(val_video_paths, val_sentences)
    train_loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=True,
        num_workers=0,  # Set to 0 for debugging
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn,
        pin_memory=True,
    )
    # Initialize models
    encoder = VivitEncoderWithTemporalModel(
        pretrained_model_name="google/vivit-b-16x2-kinetics400",
        feature_dim=768,
        hidden_dim=512,
        num_layers=2,
        temporal_model="bilstm",
        freeze_encoder=True,
        use_sliding_windows=True,
    )

    decoder = SignLanguageDecoder(
        encoder_dim=768, decoder_dim=768, bert_model_name="bert-base-german-cased"
    )

    model = SignLanguageTranslationModel(encoder, decoder).to(device)

    # Initialize optimizer and loss
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5,
        weight_decay=0.01,
        betas=(0.9, 0.999),
    )

    criterion = SignLanguageTranslationLoss(decoder.pad_token_id)
    """start_epoch = 22  # Starting from where we left off"""
    checkpoint = torch.load(str(paths.checkpoint_file))  # or 'best_model_wer.pth'
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    best_loss = checkpoint.get("loss", float("inf"))

    try:
        # Training loop
        num_epochs = 200
        best_loss = float("inf")
        total_steps = num_epochs * len(train_loader)
        scheduler = OneCycleLR(
            optimizer,
            max_lr=5e-4,
            total_steps=total_steps,
            pct_start=0.1,  # Warmup for first 10% of training
            anneal_strategy="linear",
        )
        for epoch in range(num_epochs):
            # Train
            train_loss = train_epoch(
                model=model,
                dataloader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                scheduler=scheduler,
                tokenizer=decoder.tokenizer,
                device=device,
                epoch=epoch,
            )

            # Print progress
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print(f"Training Loss: {train_loss:.4f}")

            # Validate and print example translations
            if (epoch + 1) % 5 == 0:  # Validate every 5 epochs
                predictions, targets = validate(
                    model=model,
                    dataloader=val_loader,
                    tokenizer=decoder.tokenizer,
                    device=device,
                    beam_size=5,
                )

                print("\nExample translations:")
                for pred, target in zip(predictions[:2], targets[:2]):
                    print(f"Predicted: {pred}")
                    print(f"Target: {target}")
                    print()

            # Save best model
            if train_loss < best_loss:
                best_loss = train_loss
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "loss": train_loss,
                    },
                    str(paths.checkpoint_file),
                )
                print(f"Saved new best model with loss: {best_loss:.4f}")

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        raise e
    finally:
        print("Training completed or interrupted.")


In [ ]:
if __name__ == "__main__":
    main()


In [ ]:
import traceback


### Historical setup note

The original cell installed packages interactively:

```text
pip install nltk jiwer
```

Use the central dependency files and setup guide instead.

### Historical setup note

The original cell installed packages interactively:

```text
pip install openpyxl
```

Use the central dependency files and setup guide instead.

### Historical setup note

The original cell installed packages interactively:

```text
pip install pandas
```

Use the central dependency files and setup guide instead.

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from jiwer import wer
import json
import os
from pathlib import Path


## Historical evaluation data selection

In [ ]:
excel_file = str(paths.annotations_file)
test_dir = str(paths.test_dir)


In [ ]:
# RETRIVING VIDEO_FILES, AND SENTENCES
import pandas as pd

video_files2 = glob.glob(
    os.path.join(train_dir, "*.mp4")
)  # Adjust extension as necessary

if video_files2:
    print(f"Found {len(video_files2)} video files.")
else:
    print("No video files found in the directory.")
    video_files2 = []  # Exit early if no video files found

# Step 2: Extract codes from video filenames
video_file_codes2 = [
    os.path.splitext(os.path.basename(video))[0].split("-")[-1].strip()
    for video in video_files2
]

# Step 3: Load Excel file and extract codes from the 'train' sheet
try:
    df_combined = pd.read_excel(
        excel_file, sheet_name="train"
    )  # Adjust sheet name as necessary
    df_combined["code"] = df_combined["name"].apply(lambda x: x.split("-")[-1].strip())

    # Step 4: Match video file codes with codes from the Excel file
    matching_data = df_combined[df_combined["code"].isin(video_file_codes2)]

    if not matching_data.empty:
        matched_translations2 = []
        matched_video_files2 = []

        # Iterate over video file codes to match properly
        for video_code in video_file_codes2:
            matching_row = matching_data[matching_data["code"] == video_code]
            if not matching_row.empty:
                matched_translations2.append(matching_row["translation"].iloc[0])
                matched_video_files2.append(video_code)

    else:
        # Handle cases with no matches
        unmatched_video_codes = [
            code
            for code in video_file_codes2
            if code not in df_combined["code"].tolist()
        ]

except FileNotFoundError:
    print(f"Error: Excel file '{excel_file}' not found.")
except KeyError as e:
    print(f"Error: Missing expected column in Excel file. Details: {e}")
print((matched_translations2[:5]))
print((video_files2[:5]))


### Historical setup note

The original cell installed packages interactively:

```text
pip install tabulate
```

Use the central dependency files and setup guide instead.

In [ ]:
video_paths = video_files2[:500]
sentences = matched_translations2[:500]


## Historical evaluation loop

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from jiwer import wer
import json
from pathlib import Path
from tabulate import tabulate
import traceback


### `TestSignLanguageDataset`

In [ ]:
class TestSignLanguageDataset(Dataset):
    def __init__(self, video_files, annotations):
        """
        Args:
            video_files (list): List of video file paths
            annotations (list): List of corresponding annotations/translations
        """
        assert len(video_files) == len(
            annotations
        ), "Number of videos and annotations must match"
        self.video_paths = video_paths
        self.sentences = sentences

        # Verify files exist
        for video_path in self.video_paths:
            if not Path(video_path).exists():
                print(f"Warning: Video file not found: {video_path}")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        try:
            return {"video_path": [self.video_paths[idx]], "text": self.sentences[idx]}
        except Exception as e:
            print(f"Error loading item {idx}: {str(e)}")
            print(f"Video path: {self.video_paths[idx]}")
            raise e


### `custom_collate_fn`

In [ ]:
def custom_collate_fn(batch):
    """
    Custom collate function with error handling
    """
    try:
        video_paths = []
        texts = []

        for item in batch:
            if item is not None:  # Skip None items
                video_paths.extend(item["video_path"])
                texts.append(item["text"])

        return {"video_path": video_paths, "text": texts}
    except Exception as e:
        print(f"Error in collate_fn: {str(e)}")
        traceback.print_exc()
        raise e


### `evaluate_model`

In [ ]:
@torch.no_grad()
def evaluate_model(model, test_loader, tokenizer, device, beam_size=5):
    """
    Evaluate the model using WER and BLEU scores
    """
    model.eval()
    all_predictions = []
    all_references = []
    all_video_paths = []

    # Process all test samples
    for batch_idx, batch in enumerate(tqdm(test_loader, desc="Testing")):
        try:
            if not batch["video_path"] or not batch["text"]:
                print(f"Skipping empty batch {batch_idx}")
                continue

            # Process video batch
            videos = process_video_batch(
                batch["video_path"], clip_len=32, frame_sample_rate=1
            )
            videos = videos.to(device)

            # Generate translations
            encoder_output = model.encoder(videos)
            predicted_tokens = model.decoder(encoder_output, beam_size=beam_size)

            # Convert tokens to text
            predictions = tokenizer.batch_decode(
                predicted_tokens, skip_special_tokens=True
            )

            # Store predictions, references, and video paths
            all_predictions.extend(predictions)
            all_references.extend(batch["text"])
            all_video_paths.extend(batch["video_path"])

        except Exception as e:
            print(f"Error processing batch {batch_idx}:")
            print(f"Batch content: {batch}")
            print(f"Error details: {str(e)}")
            traceback.print_exc()
            continue

    if not all_predictions or not all_references:
        raise RuntimeError("No valid predictions or references generated")

    # Calculate WER
    wer_score = wer(all_references, all_predictions)

    # Prepare references for BLEU score
    references_for_bleu = [[ref.split()] for ref in all_references]
    predictions_for_bleu = [pred.split() for pred in all_predictions]

    # Calculate BLEU score with smoothing
    smoothing = SmoothingFunction()
    bleu_score = corpus_bleu(
        references_for_bleu, predictions_for_bleu, smoothing_function=smoothing.method1
    )

    # Calculate individual word error rates
    individual_wers = []
    for ref, pred in zip(all_references, all_predictions):
        individual_wers.append(wer([ref], [pred]))

    # Create detailed comparison table
    comparison_data = []
    for i, (video_path, ref, pred, ind_wer) in enumerate(
        zip(all_video_paths, all_references, all_predictions, individual_wers), 1
    ):
        comparison_data.append(
            [
                i,
                Path(video_path[0]).name,  # Extract filename from path
                ref,
                pred,
                f"{ind_wer:.4f}",
            ]
        )

    # Print detailed comparison table
    print("\nDetailed Translation Comparison:")
    print(
        tabulate(
            comparison_data,
            headers=["#", "Video", "Reference", "Prediction", "WER"],
            tablefmt="grid",
        )
    )

    print(f"\nOverall Metrics:")
    print(f"Average WER: {wer_score:.4f}")
    print(f"BLEU Score: {bleu_score:.4f}")

    return {
        "wer": wer_score,
        "bleu": bleu_score,
        "predictions": all_predictions,
        "references": all_references,
        "video_paths": all_video_paths,
        "individual_wers": individual_wers,
        "comparison_data": comparison_data,
    }


### `main`

In [ ]:
def main():
    try:
        # Setup
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        # Your lists of video files and annotations
        video_files = video_paths[:5]

        annotations = matched_translations1[:5]
        # Initialize dataset and dataloader
        test_dataset = TestSignLanguageDataset(
            video_files=video_files, annotations=annotations
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=4,
            shuffle=False,
            num_workers=0,  # Set to 0 to avoid multiprocessing issues
            collate_fn=custom_collate_fn,
            pin_memory=True if torch.cuda.is_available() else False,
        )

        # Load model with error handling
        try:
            checkpoint = torch.load(str(paths.checkpoint_file), map_location=device)
            print("Successfully loaded checkpoint")
        except Exception as e:
            print(f"Error loading checkpoint: {str(e)}")
            raise e

        # Initialize model (use the same architecture as training)
        encoder = VivitEncoderWithTemporalModel(
            pretrained_model_name="google/vivit-b-16x2-kinetics400",
            feature_dim=768,
            hidden_dim=512,
            num_layers=2,
            temporal_model="bilstm",
            freeze_encoder=True,
            use_sliding_windows=True,
        )

        decoder = SignLanguageDecoder(
            encoder_dim=1024, bert_model_name="bert-base-german-cased"
        )

        model = SignLanguageTranslationModel(encoder, decoder).to(device)

        # Load checkpoint
        model.load_state_dict(checkpoint["model_state_dict"])
        print(
            f"Loaded model from epoch {checkpoint['epoch']} with training loss {checkpoint['loss']:.4f}"
        )

        # Evaluate model
        results = evaluate_model(
            model=model,
            test_loader=test_loader,
            tokenizer=decoder.tokenizer,
            device=device,
            beam_size=5,
        )

        # Save results
        output_dir = Path("test_results")
        output_dir.mkdir(exist_ok=True)

        # Save all results in a single JSON file
        with open(output_dir / "detailed_results.json", "w", encoding="utf-8") as f:
            json.dump(
                {
                    "overall_metrics": {"wer": results["wer"], "bleu": results["bleu"]},
                    "translations": [
                        {
                            "video": Path(video).name,
                            "reference": ref,
                            "prediction": pred,
                            "wer": wer,
                        }
                        for video, ref, pred, wer in zip(
                            results["video_paths"],
                            results["references"],
                            results["predictions"],
                            results["individual_wers"],
                        )
                    ],
                },
                f,
                indent=4,
                ensure_ascii=False,
            )

    except Exception as e:
        print(f"An error occurred in main:")
        print(str(e))
        traceback.print_exc()
        raise e


In [ ]:
if __name__ == "__main__":
    nltk.download("punkt")  # Download tokenizer data
    main()
